# Experiment No. 12: Simulate Real-World IDSS Using Classification Models (Credit Risk)

**Aim:** To design and simulate an Intelligent Decision Support System (IDSS) using machine-learning classification for credit-risk assessment, classifying applicants as Low, Medium, or High risk and providing a decision-support recommendation.

**Note:** This uses a small illustrative dataset for academic demonstration only. It is not suitable for real credit decisions.

## 1. Import libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 2. Create the illustrative credit-risk dataset

In [2]:
data = {
    'Age': [25, 32, 41, 29, 45, 36, 27, 52, 39, 31, 48, 26, 43, 35, 50, 30, 37, 42, 28, 46],
    'Income': [35000, 60000, 85000, 42000, 95000, 70000, 30000, 110000, 78000, 50000, 100000, 32000, 90000, 65000, 105000, 45000, 72000, 88000, 38000, 92000],
    'CreditScore': [610, 690, 760, 630, 790, 720, 590, 810, 750, 670, 800, 600, 770, 710, 820, 640, 730, 780, 620, 795],
    'LoanAmount': [15000, 20000, 25000, 18000, 30000, 25000, 12000, 40000, 28000, 18000, 35000, 14000, 30000, 22000, 38000, 17000, 26000, 29000, 15000, 32000],
    'Debt': [8000, 5000, 4000, 10000, 3000, 6000, 12000, 2000, 4500, 7000, 2500, 9000, 3500, 5500, 2000, 8500, 5000, 3000, 11000, 2800],
    'PreviousDefault': ['Yes', 'No', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'Yes', 'No', 'No', 'Yes', 'No'],
    'Risk': ['High', 'Medium', 'Low', 'High', 'Low', 'Low', 'High', 'Low', 'Low', 'Medium', 'Low', 'High', 'Low', 'Medium', 'Low', 'High', 'Low', 'Low', 'High', 'Low']
}
df = pd.DataFrame(data)
print("Credit Risk Dataset (first five rows):")
display(df.head())

Credit Risk Dataset (first five rows):


,Age,Income,CreditScore,LoanAmount,Debt,PreviousDefault,Risk
0,25,35000,610,15000,8000,Yes,High
1,32,60000,690,20000,5000,No,Medium
2,41,85000,760,25000,4000,No,Low
3,29,42000,630,18000,10000,Yes,High
4,45,95000,790,30000,3000,No,Low


## 3. Explore the dataset

In [3]:
print("Dataset dimensions:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
display(df.isnull().sum().to_frame('Missing Values'))
print("\nStatistical summary:")
display(df.describe(include='all').T)

Dataset dimensions: (20, 7)

Data types:
Age                 int64
Income              int64
CreditScore         int64
LoanAmount          int64
Debt                int64
PreviousDefault    object
Risk               object
dtype: object

Missing values:


,Missing Values
Age,0
Income,0
CreditScore,0
LoanAmount,0
Debt,0
PreviousDefault,0
Risk,0



Statistical summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Age,20.0,NaN,NaN,NaN,37.1,8.478456,25.0,29.75,36.5,43.5,52.0
Income,20.0,NaN,NaN,NaN,69100.0,26219.518643,30000.0,44250.0,71000.0,90500.0,110000.0
CreditScore,20.0,NaN,NaN,NaN,714.25,77.617059,590.0,637.5,725.0,782.5,820.0
LoanAmount,20.0,NaN,NaN,NaN,24450.0,8268.360749,12000.0,17750.0,25000.0,30000.0,40000.0
Debt,20.0,NaN,NaN,NaN,5715.0,3102.337828,2000.0,3000.0,5000.0,8125.0,12000.0
PreviousDefault,20,2,No,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Risk,20,3,Low,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Preprocess and encode categorical variables

In [4]:
# Encode PreviousDefault and the target Risk labels.
default_encoder = LabelEncoder()
risk_encoder = LabelEncoder()
df['PreviousDefault'] = default_encoder.fit_transform(df['PreviousDefault'])
df['Risk'] = risk_encoder.fit_transform(df['Risk'])
print("PreviousDefault encoding:", dict(zip(default_encoder.classes_, default_encoder.transform(default_encoder.classes_))))
print("Risk encoding:", dict(zip(risk_encoder.classes_, risk_encoder.transform(risk_encoder.classes_))))
display(df.head())

PreviousDefault encoding: {'No': np.int64(0), 'Yes': np.int64(1)}
Risk encoding: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}


,Age,Income,CreditScore,LoanAmount,Debt,PreviousDefault,Risk
0,25,35000,610,15000,8000,1,0
1,32,60000,690,20000,5000,0,2
2,41,85000,760,25000,4000,0,1
3,29,42000,630,18000,10000,1,0
4,45,95000,790,30000,3000,0,1


## 5. Separate features and target

In [5]:
feature_columns = ['Age', 'Income', 'CreditScore', 'LoanAmount', 'Debt', 'PreviousDefault']
X = df[feature_columns]
y = df['Risk']
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (20, 6)
Target shape: (20,)


## 6. Split into training and testing sets (80:20)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 16
Testing samples: 4


## 7. Train the Decision Tree classifier

In [7]:
model = DecisionTreeClassifier(criterion='entropy', max_depth=4, random_state=42)
model.fit(X_train, y_train)
print("Decision Tree classifier trained successfully.")

Decision Tree classifier trained successfully.


## 8. Predict risk categories for the test set

In [8]:
y_pred = model.predict(X_test)
test_results = pd.DataFrame({
    'Actual Risk': risk_encoder.inverse_transform(y_test),
    'Predicted Risk': risk_encoder.inverse_transform(y_pred)
}, index=y_test.index).sort_index()
display(test_results)

,Actual Risk,Predicted Risk
1,Medium,Medium
8,Low,Low
11,High,High
12,Low,Low


## 9. Evaluate the model

In [9]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")
print("\nConfusion Matrix (class order: " + ", ".join(risk_encoder.classes_) + "):")
print(confusion_matrix(y_test, y_pred, labels=risk_encoder.transform(risk_encoder.classes_)))
print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    labels=risk_encoder.transform(risk_encoder.classes_),
    target_names=risk_encoder.classes_,
    zero_division=0
))

Model Accuracy: 100.00%

Confusion Matrix (class order: High, Low, Medium):
[[1 0 0]
 [0 2 0]
 [0 0 1]]

Classification Report:
              precision    recall  f1-score   support

        High       1.00      1.00      1.00         1
         Low       1.00      1.00      1.00         2
      Medium       1.00      1.00      1.00         1

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4



## 10. Enter details for a new applicant

In [10]:
new_applicant = pd.DataFrame({
    'Age': [38],
    'Income': [70000],
    'CreditScore': [735],
    'LoanAmount': [25000],
    'Debt': [6000],
    'PreviousDefault': [default_encoder.transform(['No'])[0]]
})
display(new_applicant)

,Age,Income,CreditScore,LoanAmount,Debt,PreviousDefault
0,38,70000,735,25000,6000,0


## 11. Predict the new applicant's credit risk

In [11]:
prediction = model.predict(new_applicant)
predicted_risk = risk_encoder.inverse_transform(prediction)[0]
print("Predicted Credit Risk:", predicted_risk)

Predicted Credit Risk: Low


## 12. Generate the decision-support recommendation

In [12]:
if predicted_risk == 'Low':
    recommendation = 'Low-risk applicant. The application may be considered for approval.'
elif predicted_risk == 'Medium':
    recommendation = 'Medium-risk applicant. Additional verification is recommended.'
else:
    recommendation = 'High-risk applicant. Detailed risk assessment is recommended.'
print("Decision Support Recommendation:")
print(recommendation)
print("\nNote: This is an educational IDSS demonstration, not a real credit assessment tool.")

Decision Support Recommendation:
Low-risk applicant. The application may be considered for approval.

Note: This is an educational IDSS demonstration, not a real credit assessment tool.


## Result
The Intelligent Decision Support System was implemented using a Decision Tree classification model in Python. The system explores and preprocesses credit-risk data, trains and evaluates the classifier using accuracy, a confusion matrix, precision, recall, and F1-score, then predicts the risk category of a new applicant and generates a decision-support recommendation.